# Download Models To Google Drive

本 Notebook 用于在 Google Colab 中挂载 Google Drive，创建 models 目录，并将 Stable Diffusion 3.5 Medium 与 InSPyReNet 权重保存到 Drive。

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    raise RuntimeError('该 Notebook 设计用于 Google Colab，请在 Colab 中运行。')

print('=' * 80)
print('初始化 Google Drive 模型下载环境')
print('=' * 80)

drive.mount('/content/drive', force_remount=False)

WORK_DIR = Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG_WM_Colab')
DRIVE_MODELS_DIR = DRIVE_ROOT / 'models'
DRIVE_SD35_DIR = DRIVE_MODELS_DIR / 'stable-diffusion-3.5-medium'
DRIVE_HF_CACHE_DIR = DRIVE_MODELS_DIR / 'huggingface_cache'
DRIVE_INSPYRENET_DIR = DRIVE_MODELS_DIR / 'inspyrenet'
DRIVE_INSPYRENET_WEIGHT_PATH = DRIVE_INSPYRENET_DIR / 'ckpt_base.pth'

for path in [DRIVE_ROOT, DRIVE_MODELS_DIR, DRIVE_SD35_DIR, DRIVE_HF_CACHE_DIR, DRIVE_INSPYRENET_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'WORK_DIR={WORK_DIR}')
print(f'DRIVE_ROOT={DRIVE_ROOT}')
print(f'DRIVE_MODELS_DIR={DRIVE_MODELS_DIR}')

subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=False)
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=False)
print('✓ 基础依赖已就绪')

In [ ]:
from huggingface_hub import HfApi, login, notebook_login, snapshot_download

HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
USE_NOTEBOOK_LOGIN = False
MODEL_ID = 'stabilityai/stable-diffusion-3.5-medium'

os.environ['HF_HOME'] = str(DRIVE_HF_CACHE_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(DRIVE_HF_CACHE_DIR)

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ 已使用 HF_TOKEN 登录 Hugging Face')
elif USE_NOTEBOOK_LOGIN:
    notebook_login()
else:
    print('ℹ 未显式登录 Hugging Face；如果模型访问失败，请设置 HF_TOKEN 或开启 USE_NOTEBOOK_LOGIN。')

api = HfApi()
_ = api.model_info(MODEL_ID)
print(f'✓ 模型可访问: {MODEL_ID}')

existing_files = list(DRIVE_SD35_DIR.rglob('*')) if DRIVE_SD35_DIR.exists() else []
if any(path.is_file() for path in existing_files):
    print(f'✓ Google Drive 中已存在 SD3.5 文件，跳过重复下载: {DRIVE_SD35_DIR}')
else:
    print(f'开始下载 SD3.5 到 Google Drive: {DRIVE_SD35_DIR}')
    snapshot_path = snapshot_download(
        repo_id=MODEL_ID,
        cache_dir=str(DRIVE_HF_CACHE_DIR),
        local_dir=str(DRIVE_SD35_DIR),
        local_dir_use_symlinks=False,
        local_files_only=False,
    )
    print(f'✓ SD3.5 下载完成: {snapshot_path}')

print(f'✓ SD3.5 Drive 目录: {DRIVE_SD35_DIR}')

In [ ]:
import urllib.request

INSPYRENET_URL = 'https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth'

if DRIVE_INSPYRENET_WEIGHT_PATH.exists() and DRIVE_INSPYRENET_WEIGHT_PATH.stat().st_size > 0:
    print(f'✓ InSPyReNet 权重已存在，跳过下载: {DRIVE_INSPYRENET_WEIGHT_PATH}')
else:
    temp_path = DRIVE_INSPYRENET_WEIGHT_PATH.with_suffix('.pth.downloading')
    if temp_path.exists():
        temp_path.unlink()
    print(f'开始下载 InSPyReNet 权重到 Google Drive: {INSPYRENET_URL}')
    urllib.request.urlretrieve(INSPYRENET_URL, str(temp_path))
    if not temp_path.exists() or temp_path.stat().st_size <= 0:
        raise RuntimeError(f'InSPyReNet 权重下载失败: {temp_path}')
    temp_path.replace(DRIVE_INSPYRENET_WEIGHT_PATH)
    print(f'✓ InSPyReNet 权重下载完成: {DRIVE_INSPYRENET_WEIGHT_PATH}')

print('=' * 80)
print('Google Drive 模型准备完成')
print('=' * 80)
print(f'SD3.5 目录: {DRIVE_SD35_DIR}')
print(f'InSPyReNet 权重: {DRIVE_INSPYRENET_WEIGHT_PATH}')